# N5 — Liquidity Actions

## Decision question

Which combination of collections, supplier, inventory, and facility actions
protects liquidity at an acceptable direct cost and relationship risk?

All effects occur on explicit dates and flow back into the cash forecast.


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.1.0-beta.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    reveal_team_shock,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N5')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']} (model cache: {'hit' if summary['model_cache_hit'] else 'rebuilt'})")


## Compare action packages


In [ ]:
scenarios = pd.read_csv(OUTPUT_DIR / 'N5_action_scenarios.csv')
display(scenarios)
viz.action_scenarios(scenarios, manifest['minimum_liquidity'], OUTPUT_DIR)


## Inspect your selected package


In [ ]:
selected = pd.read_csv(OUTPUT_DIR / 'N5_selected_action_forecast.csv', parse_dates=['date'])
realistic = pd.read_csv(OUTPUT_DIR / 'N4_realistic_forecast.csv', parse_dates=['date'])
viz.forecast_chart(
    selected,
    manifest['minimum_liquidity'],
    OUTPUT_DIR,
    'N5_selected_action_forecast.png',
    'Selected actions versus no-action realistic case',
    comparison=realistic,
)
display(selected[['day', 'receipts', 'total_outflows', 'closing_cash', 'below_minimum']])


## Freeze the initial recommendation

Record the package before opening the event envelope. The next evidence is new;
do not rewrite the original decision after seeing it.


In [ ]:
initial_decision = dict(DECISIONS)
(OUTPUT_DIR / 'N5_pre_shock_decision.json').write_text(
    json.dumps(initial_decision, indent=2), encoding='utf-8'
)
print('Initial package frozen. Open the event envelope only when instructed.')


## Incoming event — revise or defend


In [ ]:
if not DECISIONS['shock_revealed']:
    DECISIONS['scenario_variant'] = reveal_team_shock(DECISIONS['team_name'])
    DECISIONS['shock_revealed'] = True
    decision_file.write_text(json.dumps(DECISIONS, indent=2), encoding='utf-8')

event = manifest['scenario_variants'][DECISIONS['scenario_variant']]
display(Markdown(f"### EVENT ENVELOPE: {event['label']}"))
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
stress = pd.read_csv(OUTPUT_DIR / 'N5_execution_stress.csv')
display(stress)
viz.execution_stress(stress, manifest['minimum_liquidity'], OUTPUT_DIR)


## Team decision

Choose the lowest-cost package you can defend. Record what must go right, what
could fail, and which fallback is pre-authorized. Edit the consequential choices
below and rerun; do not revise merely to make every metric green.


In [ ]:
DECISIONS.update({
    'execution_case': 'expected',  # expected, downside, failed
    'collection_strategy': DECISIONS['collection_strategy'],
    'payables_extension_days': DECISIONS['payables_extension_days'],
    'inventory_release_pct': DECISIONS['inventory_release_pct'],
    'facility_draw': DECISIONS['facility_draw'],
})
decision_file.write_text(json.dumps(DECISIONS, indent=2), encoding='utf-8')
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
display(pd.read_csv(OUTPUT_DIR / 'N5_execution_stress.csv'))
print('Post-shock package recorded in the decision ledger.')


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
